In [28]:
import pandas as pd
import numpy as np

In [29]:
df_bi = pd.read_csv('../data/RawData/telco_customer_data_v2.csv')
df_bi.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST00001,Male,0,No,Yes,3.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,68.61,205.83,Yes
1,CUST00002,Male,1,Yes,No,2.0,Yes,Yes,DSL,No,...,No internet service,Yes,NaN,No,One year,Yes,Bank transfer (automatic),23.15,46.3,No
2,CUST00003,Female,No,No,No,42.0,Yes,Yes,DSL,No,...,No,NaN,Yes,Yes,Month-to-month,No,Electronic check,42.63,1790.46,Yes
3,CUST00004,Female,0,No,Yes,40.0,Yes,Yes,Fiber optic,No,...,Yes,No,No,No internet service,Month-to-month,No,Electronic check,75.04,3001.6,No
4,CUST00005,Male,Yes,Yes,Yes,17.0,Yes,NaN,Fiber optic,Yes,...,Yes,No,No internet service,No,Two year,Yes,Electronic check,22.38,380.46,Yes


In [30]:
# ============ 1. STANDARDIZE TEXT ============
# Strip whitespace from all object columns
object_cols = df_bi.select_dtypes(include='object').columns
for col in object_cols:
    df_bi[col] = df_bi[col].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan})

# for col in df_bi.select_dtypes(include='object').columns:
#     print(f"Unique values in '{col}': {df_bi[col].unique()}")


In [31]:
# ============ 2. FIX CATEGORICAL CONSISTENCY ============
    
# Gender: Standardize to Male/Female
gender_map = {
        'male': 'Male', 'm': 'Male', 'man': 'Male',
        'female': 'Female', 'f': 'Female', 'woman': 'Female'
}

df_bi['gender'] = df_bi['gender'].str.lower()

df_bi['gender'] = df_bi['gender'].map(gender_map).fillna(df_bi['gender'])
df_bi.loc[~df_bi['gender'].isin(['Male', 'Female']), 'gender'] = np.nan

    # Churn: Standardize to Yes/No
churn_map = {
'yes': 'Yes', 'y': 'Yes', 'churned': 'Yes', '1': 'Yes',
'no': 'No', 'n': 'No', 'no churn': 'No', '0': 'No'
}
df_bi['Churn'] = df_bi['Churn'].astype(str).str.lower().str.strip().map(churn_map)
df_bi['Churn'] = df_bi['Churn'].fillna('Unknown')

# Contract: Fix abbreviations
df_bi['Contract'] = df_bi['Contract'].replace({'M-M': 'Month-to-month', 'month-to-month': 'Month-to-month'})
    
# Service columns: Fix Y/True variations
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                   'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in service_cols:
    df_bi[col] = df_bi[col].replace({
                'Y': 'Yes', 'True': 'Yes', 'true': 'Yes',
                'N': 'No', 'False': 'No', 'false': 'No'
            })
    
# SeniorCitizen: Ensure numeric 0/1
df_bi['SeniorCitizen'] = pd.to_numeric(
        df_bi['SeniorCitizen'].replace({'Yes': 1, 'No': 0, 'not senior': 0}), 
        errors='coerce'
    )    



In [32]:
print(df_bi['SeniorCitizen'].unique())

[ 0.  1. nan]


In [33]:
# ============ 3. FIX NUMERICAL ISSUES ============
    
# TotalCharges: Ensure numeric, handle negatives
df_bi['TotalCharges'] = pd.to_numeric(df_bi['TotalCharges'], errors='coerce')
df_bi.loc[df_bi['TotalCharges'] < 0, 'TotalCharges'] = np.nan
    
# tenure: Ensure non-negative integer
df_bi['tenure'] = pd.to_numeric(df_bi['tenure'], errors='coerce')
df_bi.loc[df_bi['tenure'] < 0, 'tenure'] = np.nan
df_bi['tenure'] = df_bi['tenure'].round(0)
    
print(df_bi['SeniorCitizen'].unique())

[ 0.  1. nan]


In [34]:
# ============ 4. HANDLE MISSING VALUES (BI Strategy) ============
    
# Create "Unknown" categories for filters
categorical_missing = ['gender', 'Partner', 'Dependents', 'PaymentMethod', 'Contract']
for col in categorical_missing:
    if col in df_bi.columns and df_bi[col].isna().any():
        df_bi[col] = df_bi[col].fillna('Unknown')
    
# Service columns: missing likely means "No internet service"
for col in service_cols:
    if col in df_bi.columns:
        df_bi[col] = df_bi[col].fillna('No internet service')

# Case 1: If no phone service → enforce correct value
df_bi.loc[df_bi["PhoneService"] == "No", "MultipleLines"] = "No phone service"

# Case 2: If phone service exists → fill missing with "No"
df_bi.loc[df_bi["PhoneService"] == "Yes", "MultipleLines"] = (
    df_bi.loc[df_bi["PhoneService"] == "Yes", "MultipleLines"]
    .fillna("No")
)

# Delete rows where specific columns have "Unknown"
df_bi = df_bi[df_bi['Churn'] != 'Unknown']
df_bi = df_bi.dropna(subset=['MonthlyCharges'])
df_bi = df_bi.dropna(subset=['SeniorCitizen'])

df_bi["tenure"] = df_bi.groupby("Churn")["tenure"].transform(
    lambda x: x.fillna(x.median())
)

mask = df_bi["TotalCharges"].isna() & df_bi["tenure"].notna() & df_bi["MonthlyCharges"].notna()
df_bi.loc[mask, "TotalCharges"] = df_bi.loc[mask, "tenure"] * df_bi.loc[mask, "MonthlyCharges"]

In [35]:
df_bi['PaymentMethod'] = df_bi['PaymentMethod'].replace({
    'BANK TRANSFER': 'Bank transfer (automatic)',
    'bank transfer': 'Bank transfer (automatic)',
    'Bank Transfer': 'Bank transfer (automatic)',
    'Credit Card': 'Credit card (automatic)',
    'credit card': 'Credit card (automatic)',
    'Electronic Check': 'Electronic check',
    'Mailed Check': 'Mailed check'
})

print(df_bi['PaymentMethod'].value_counts())

PaymentMethod
Electronic check             27433
Bank transfer (automatic)    13998
Mailed check                 13732
Credit card (automatic)      10228
Unknown                       3515
Name: count, dtype: int64


In [36]:
# Quick validation
print(f" Cleaned shape: {df_bi.shape}")
print(f"\n Churn distribution:\n{df_bi['Churn'].value_counts()}")
print(f"\n Gender distribution:\n{df_bi['gender'].value_counts()}")
print(f"\n Tenure negative values: {(df_bi['tenure'] < 0).sum()}")
print(f"\n TotalCharges negative values: {(df_bi['TotalCharges'] < 0).sum()}")

 Cleaned shape: (68906, 21)

 Churn distribution:
Churn
No     36617
Yes    32289
Name: count, dtype: int64

 Gender distribution:
gender
Female     34258
Male       33914
Unknown      734
Name: count, dtype: int64

 Tenure negative values: 0

 TotalCharges negative values: 0


In [37]:
for col in df_bi.select_dtypes(include='object').columns:
    print(f"Unique values in '{col}': {df_bi[col].unique()}")

Unique values in 'customerID': ['CUST00001' 'CUST00002' 'CUST00003' ... 'CUST69998' 'CUST69999'
 'CUST70000']
Unique values in 'gender': ['Male' 'Female' 'Unknown']
Unique values in 'Partner': ['No' 'Yes' 'Unknown']
Unique values in 'Dependents': ['Yes' 'No' 'Unknown']
Unique values in 'PhoneService': ['Yes' 'No']
Unique values in 'MultipleLines': ['Yes' 'No' 'No phone service']
Unique values in 'InternetService': ['No' 'DSL' 'Fiber optic']
Unique values in 'OnlineSecurity': ['No internet service' 'No' 'Yes']
Unique values in 'OnlineBackup': ['No internet service' 'No' 'Yes']
Unique values in 'DeviceProtection': ['No internet service' 'No' 'Yes']
Unique values in 'TechSupport': ['No internet service' 'Yes' 'No']
Unique values in 'StreamingTV': ['No internet service' 'Yes' 'No']
Unique values in 'StreamingMovies': ['No internet service' 'No' 'Yes']
Unique values in 'Contract': ['Month-to-month' 'One year' 'Two year']
Unique values in 'PaperlessBilling': ['No' 'Yes']
Unique values in 'Pa

In [38]:
# Run these to confirm cleaning worked
assert df_bi['gender'].isin(['Male', 'Female', 'Unknown']).all(), "Gender has invalid values"
assert df_bi['Churn'].isin(['Yes', 'No', 'Unknown']).all(), "Churn has invalid values"
assert (df_bi['tenure'].dropna() >= 0).all(), "tenure still has negative values"
assert (df_bi['TotalCharges'].dropna() >= 0).all(), "TotalCharges still has negative values"
assert df_bi['Contract'].isin(['Month-to-month', 'One year', 'Two year', 'Unknown']).all(), "Contract has invalid values"

print(" All validation checks passed!")

 All validation checks passed!


In [39]:
# # CSV
df_bi.to_csv("cleaned_churn.csv", index=False)

# # Excel
# df_bi.to_excel("cleaned_churn_data.xlsx", index=False)